# Lab 17 · Uczciwe porównanie

Przed wykonaniem tego laboratorium przeczytaj wykłady [D14.1 „Czytaj metryki razem z ich ograniczeniami”](../lectures/d14-metrics.html), [D14.2 „Porównaj trening i walidację”](../lectures/d14-overfit.html) i [D14.3 „Powtórz prognozę w kilku tygodniach”](../lectures/d14-backtest.html).

Pamiętaj, że do tego labu potrzebne jest środowisko z labu 9: projekt practice z bibliotekami sekcji C. Uruchom Jupyter tak samo jak tam.

Pod opisem każdego zadania znajduje się komórka kodu do uzupełnienia, a pod nią komórka sprawdzenia. Jeżeli zadanie nie jest jeszcze rozwiązane, sprawdzenie zatrzyma się i wyświetli czerwony komunikat. To normalne: komunikat mówi, co trzeba poprawić.

Najpierw przewidź wynik, potem uruchom kod. Podpowiedzi otwieraj pojedynczo.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-17-uczciwe-porownanie.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)

## D14.1 · Cztery miary w pliku practice/metrics.py

Po wykładzie D14.1 otwórz `practice/metrics.py` i uzupełnij `error_summary(actual, forecast)` według docstringa:

- wynik to słownik z polami `mae`, `rmse`, `wape` i `bias`,
- `bias` to suma różnic prognoza minus obserwacja,
- `wape` to suma wielkości błędów podzielona przez sumę obserwacji,
- gdy suma obserwacji wynosi zero, `wape` jest `None` zamiast dzielenia,
- różne długości wejść zgłaszają `ValueError`,
- wejścia pozostają bez zmian,
- sygnatura i adnotacje typów zostają takie, jak w pliku; dodaj `import numpy as np`, jeśli używasz tablic.

Po zapisaniu pliku wykonaj gotową komórkę poniżej.

In [ ]:
import runpy
metrics_path = lab_dir.parent / "practice" / "metrics.py"
assert metrics_path.is_file(), "Brakuje practice/metrics.py w repozytorium."
error_summary = runpy.run_path(str(metrics_path))["error_summary"]

In [ ]:
import math
actual_a, forecast_a = [2.0, 8.0], [4.0, 6.0]
summary_a = error_summary(actual_a, forecast_a)
assert isinstance(summary_a, dict) and set(summary_a) == {"mae", "rmse", "wape", "bias"}, "D14.1: zwróć słownik z czterema polami mae, rmse, wape i bias."
assert all(isinstance(summary_a[key], (int, float, np.floating)) for key in ("mae", "rmse", "wape", "bias")), "D14.1: dla dwóch dni bez zerowej sumy wszystkie cztery pola są liczbami."
assert math.isclose(summary_a["mae"], 2.0) and math.isclose(summary_a["rmse"], 2.0), "D14.1: błędy 2 i -2 dają MAE 2 i RMSE 2."
assert math.isclose(summary_a["wape"], 0.4) and math.isclose(summary_a["bias"], 0.0), "D14.1: WAPE to 4 / 10, a bias to suma prognoza minus obserwacja, czyli 0."
assert actual_a == [2.0, 8.0] and forecast_a == [4.0, 6.0], "D14.1: nie zmieniaj list wejściowych."
summary_b = error_summary([0.0, 10.0], [2.0, 8.0])
assert math.isclose(summary_b["mae"], 2.0) and math.isclose(summary_b["wape"], 0.4) and math.isclose(summary_b["bias"], 0.0), "D14.1: dzień z zerową sprzedażą pozostaje w ocenie."
summary_c = error_summary([0.0, 0.0], [1.0, 0.0])
assert summary_c["wape"] is None and math.isclose(summary_c["mae"], 0.5) and math.isclose(summary_c["bias"], 1.0), "D14.1: przy zerowej sumie obserwacji wape to None; mae i bias nadal są liczbami."
try:
    error_summary([1.0, 2.0], [1.0])
except ValueError:
    pass
else:
    raise AssertionError("D14.1: różne długości mają zgłaszać ValueError; nie oceniaj niedopasowanych dni.")
print("D14.1: cztery miary i przypadek zerowego mianownika działają zgodnie z umową.")

<details><summary>Podpowiedź 1</summary>

Zamień wejścia na tablice np.asarray i policz różnice forecast minus actual raz; z nich wynikają wszystkie pola.

</details>

<details><summary>Podpowiedź 2</summary>

Sprawdź długości przed obliczeniami i zgłoś ValueError. Dla wape najpierw sprawdź sumę actual, a potem dziel.

</details>

## D14.1 · Testy metryk

Otwórz `practice/test_metrics.py` i zastąp trzy placeholdery własnymi testami. Dodaj na początku `from metrics import error_summary` oraz `import pytest`, jeśli używasz `pytest.approx` i `pytest.raises`. Oczekiwane liczby policz ręcznie na dwóch dniach, nie wywołaniem tej samej funkcji.

Polecenia wykonaj w terminalu z folderu practice:

```bash
uv run ruff format metrics.py test_metrics.py
uv run ruff check metrics.py test_metrics.py
uv run mypy --disallow-untyped-defs metrics.py
uv run python -m pytest -q test_metrics.py
```

Oczekujesz `3 passed`. Jeżeli w B7.2 masz workflow CI, dodaj oba pliki do jego kroków, tak jak w labie 13.

## D14.2 · Który wariant wybrać

Po wykładzie D14.2 słownik `variants` podaje dla trzech wariantów modelu parę (MAE treningu, MAE walidacji). W `chosen` zapisz nazwę wariantu do wyboru, w `overfit_variant` nazwę wariantu dopasowanego do szczegółów treningu, a w `underfit_variant` nazwę wariantu zbyt prostego. Test końcowy pozostaje zamknięty i nie bierze udziału w wyborze.

In [ ]:
variants = {"a": (3.1, 3.3), "b": (1.9, 2.2), "c": (0.2, 2.9)}
chosen = None
overfit_variant = None
underfit_variant = None

In [ ]:
assert chosen == "b", "D14.2: wybieraj po MAE walidacji, nie treningu."
assert overfit_variant == "c", "D14.2: wariant z bardzo niskim błędem treningu i dużo wyższym walidacji dopasował się do szczegółów."
assert underfit_variant == "a", "D14.2: wariant z dużym błędem w obu okresach jest zbyt prosty."
print("D14.2: wybór oparty na walidacji; obie pomyłki nazwane.")

<details><summary>Podpowiedź 1</summary>

Druga liczba w parze to walidacja.

</details>

<details><summary>Podpowiedź 2</summary>

Overfitting to duża przepaść między treningiem a walidacją; underfitting to wysoki błąd w obu.

</details>

## Dane projektu i cecha z C12

Gotowa komórka odtwarza tabele z labu 14: `rows` z cechą `lag1` oraz podział `train` i `valid` według `cutoff`. W tym labie backtest używa wyłącznie dni treningowych do 11 lutego; tydzień walidacji i holdout zostają poza foldami.

In [ ]:
sales = pd.read_csv(lab_dir / "data" / "project-development.csv", parse_dates=["day"])
assert sales.shape == (49, 2), "Oczekiwano 49 dni i dwóch kolumn."
cutoff = pd.Timestamp("2024-02-12")
with_lag = sales.assign(lag1=sales["sale"].shift(1))
rows = with_lag[with_lag["lag1"].notna()]
train = rows[rows["day"] < cutoff]
valid = rows[rows["day"] >= cutoff]
assert len(train) == 41 and len(valid) == 7
print("Trening:", train["day"].min().date(), "–", train["day"].max().date(), "| walidacja:", valid["day"].min().date(), "–", valid["day"].max().date())

## D14.3 · Trzy datowane podziały

Po wykładzie D14.3 w `folds` zapisz listę trzech słowników z polami `train_end`, `valid_start` i `valid_end` jako napisy `RRRR-MM-DD`. Oceniane tygodnie to 22–28 stycznia, 29 stycznia–4 lutego i 5–11 lutego 2024; trening każdego foldu kończy się dzień przed jego tygodniem oceny. Każdy fold ma siedem dni oceny.

In [ ]:
folds = None

In [ ]:
assert isinstance(folds, list) and len(folds) == 3 and all(isinstance(fold, dict) for fold in folds), "D14.3: folds to lista trzech słowników."
assert folds == [{'train_end': '2024-01-21', 'valid_start': '2024-01-22', 'valid_end': '2024-01-28'}, {'train_end': '2024-01-28', 'valid_start': '2024-01-29', 'valid_end': '2024-02-04'}, {'train_end': '2024-02-04', 'valid_start': '2024-02-05', 'valid_end': '2024-02-11'}], "D14.3: trening kończy się dzień przed tygodniem oceny; trzy tygodnie od 22 stycznia do 11 lutego."
print("D14.3: trzy foldy mają jawne daty i nie sięgają tygodnia walidacji C12.")

<details><summary>Podpowiedź 1</summary>

Każdy słownik ma trzy daty; valid_start to dzień po train_end.

</details>

<details><summary>Podpowiedź 2</summary>

Napisy dat porównasz później z kolumną day; format RRRR-MM-DD jest tu wymagany.

</details>

## D14.3 · Świeży fit w każdym foldzie

W `fold_results` zapisz listę trzech słowników z polami `model_mae` i `baseline_mae`. Dla każdego foldu dopasuj nową `LinearRegression` na wierszach `rows` z dniem do `train_end` włącznie, policz MAE prognoz na dniach od `valid_start` do `valid_end` i MAE baseline „jak wczoraj” na tych samych dniach. W `differences` zapisz listę `model_mae - baseline_mae` dla kolejnych foldów, a w `wins` liczbę foldów, w których model ma niższe MAE. Remis nie jest wygraną. Możesz użyć własnej `error_summary`.

In [ ]:
fold_results = None
differences = None
wins = None

In [ ]:
import math
assert isinstance(fold_results, list) and len(fold_results) == 3, "D14.3: jeden słownik wyników na fold."
expected = [{'model_mae': 2.117186977739153, 'baseline_mae': 2.8785714285714277}, {'model_mae': 2.9940523565982837, 'baseline_mae': 3.2071428571428577}, {'model_mae': 2.246250190329963, 'baseline_mae': 2.9214285714285717}]
for got, wanted in zip(fold_results, expected):
    assert isinstance(got, dict) and math.isclose(float(got["model_mae"]), wanted["model_mae"], rel_tol=1e-6), "D14.3: dopasuj model od nowa na wierszach do train_end; oceniaj siedem dni foldu."
    assert math.isclose(float(got["baseline_mae"]), wanted["baseline_mae"], rel_tol=1e-6), "D14.3: baseline to lag1 na tych samych siedmiu dniach."
assert isinstance(differences, list) and np.allclose(differences, [-0.7613844508322747, -0.213090500544574, -0.6751783810986085]), "D14.3: differences to model minus baseline w kolejności foldów."
assert wins == sum(d < 0 for d in differences), "D14.3: wins liczy foldy z niższym MAE modelu; remis nie jest wygraną."
print("D14.3: wyniki sparowane po foldach. Wygrane modelu:", wins, "z 3.")

<details><summary>Podpowiedź 1</summary>

Pętla po folds: dwie maski po kolumnie day, nowy model w każdym obiegu.

</details>

<details><summary>Podpowiedź 2</summary>

differences policz z pól każdego słownika wyników. wins to suma warunków różnica < 0; zero nie jest wygraną.

</details>

## Sprawdź rezultat

Sprawdzenia przechodzą po restarcie kernela, a testy metryk w terminalu. W trzech wcześniejszych tygodniach model wygrywa z baseline, a w tygodniu walidacji z C12 przegrał. Do przemyślenia: co to mówi o wielkości próbki i dlaczego nie uśredniasz różnych tygodni między sobą? Czy średnia z trzech foldów jest przedziałem ufności? Nie musisz spisywać odpowiedzi.